![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Week 2, Day 3 -- Lab 4: Fine-tune a Pretrained Model

The most common move in transfer learning: take a model trained on millions of
images, **replace its final classifier** with a new one for *your* classes, and
train only that part. It's fast and works even with little data.

Here we teach a pretrained ResNet to tell **ants from bees** using only a few
hundred photos.

## 1) Setup

In [ ]:
import torch
import torch.nn as nn
import urllib.request, zipfile
from torchvision import datasets
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader

## 2) Load a pretrained model and freeze it

Freezing keeps everything it already learned and won't change it.

In [ ]:
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
for p in model.parameters():
    p.requires_grad = False              # freeze the backbone

## 3) Replace the classifier

ResNet was built for 1000 classes. We swap its last layer for one with **2** outputs (ants, bees). Only this new layer will train.

In [ ]:
model.fc = nn.Linear(model.fc.in_features, 2)    # new, trainable classifier
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

## 4) Get the data (~45 MB)

A few hundred ant and bee photos, already split into train/val folders.

In [ ]:
urllib.request.urlretrieve("https://download.pytorch.org/tutorial/hymenoptera_data.zip", "data.zip")
zipfile.ZipFile("data.zip").extractall()

In [ ]:
prep = weights.transforms()
train_dl = DataLoader(datasets.ImageFolder("hymenoptera_data/train", prep), batch_size=16, shuffle=True)
val_dl   = DataLoader(datasets.ImageFolder("hymenoptera_data/val", prep), batch_size=16)

## 5) Train only the new classifier

In [ ]:
opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)   # only the new layer
loss_fn = nn.CrossEntropyLoss()
model.train()
for epoch in range(3):
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss_fn(model(x), y).backward()
        opt.step()
    print(f"epoch {epoch + 1} done")

## 6) Check accuracy on unseen images

In [ ]:
model.eval(); correct = total = 0
with torch.no_grad():
    for x, y in val_dl:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
print(f"accuracy: {correct / total:.1%}")

## Try it yourself

You trained only the final layer on a few hundred photos and still got strong
accuracy -- because the frozen backbone already knows how to *see*. Try more
epochs, or unfreeze a few more layers.

### *Contributed By: Sattam Altwaim*